# Test notebook

In [2]:
import pypsa
import numpy as np
import pandas as pd


In [4]:
import matplotlib.pyplot as plt

import pypsa

%matplotlib inline
plt.rc("figure", figsize=(8, 8))

In [2]:
n = pypsa.Network()

In [3]:
#add three buses
n_buses = 3

for i in range(n_buses):
    n.add("Bus", "My bus {}".format(i),  v_nom=20.)

In [4]:
#add three lines in a ring
for i in range(n_buses):
    n.add("Line", "My line {}".format(i),
                bus0="My bus {}".format(i),
                bus1="My bus {}".format((i+1)%3),
                x=0.1,
                r=0.01)

#add a generator at bus 0
n.add("Generator", "My gen",
            bus="My bus 0",
            p_set=150,
           p_nom_extendable=False,
          control='PQ'
)


#add a load at bus 1
n.add("Load", "My load",
            bus="My bus 1",
            p_set=150)

In [ ]:
n.plot()

In [5]:
n.pf()

INFO:pypsa.pf:Performing non-linear load-flow on AC sub-network SubNetwork 0 for snapshots Index(['now'], dtype='object')
INFO:pypsa.pf:Newton-Raphson solved in 3 iterations with error of 0.000000 in 0.029200 seconds


{'converged':         0
 now  True, 'error':                 0
 now  5.528689e-11, 'n_iter':      0
 now  3}

In [5]:
network = pypsa.Network()

# Add a bus
network.add("Bus", "bus0")

# Add a generator with expandable capacity
network.add("Generator", "gen0",
           bus="bus0",
           p_nom_extendable=True,  # Allow capacity expansion
           capital_cost=100,  # Cost per MW of capacity
           marginal_cost=50)  # Cost in €/MWh

# Add a load
network.add("Load", "load0",
           bus="bus0",
           p_set=50)  # Power demand in MW

# Optimize the network for capacity expansion
network.optimize(solver_name='cbc')



# Print optimized generator capacity
print("Optimized generator capacity (MW):")
print(network.generators.p_nom_opt)

Index(['bus0'], dtype='object', name='Bus')
INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.10 
Build Date: Aug  1 2023 

command line - cbc -printingOptions all -import /var/folders/pw/xvsmvk6n5b94yztdk7h497dw0000gn/T/linopy-problem-kk5vluq3.lp -solve -solu /var/folders/pw/xvsmvk6n5b94yztdk7h497dw0000gn/T/linopy-solve-lh1t_e11.sol (default strategy 1)
Option for printingOptions changed from normal to all
Presolve 0 (-4) rows, 0 (-2) columns and 0 (-5) elements
Empty problem - 0 rows, 0 columns and 0 elements
Optimal - objective value 7500
After Postsolve, objective 7500, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 7500 - 0 iterations time 0.012, Presolve 0.01
Total time (CPU seconds):       0.04   (Wallclock seconds):       0.01


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 2 primals, 4 duals
Objective: 7.50e+

Optimized generator capacity (MW):
Generator
gen0    50.0
Name: p_nom_opt, dtype: float64


In [8]:
network

Network 

In [14]:
n.optimize()

AttributeError: 'Network' object has no attribute 'optimize'

In [ ]:
n.plot()

### Load data

In [23]:
df_load = pd.read_csv('data/load_data.csv', parse_dates=["Datetime"],index_col='Datetime')

In [18]:
df_solar = pd.read_csv('data/solar_data.csv', parse_dates=["Datetime"],index_col='Datetime')

In [15]:
df_market = pd.read_csv('data/market_data.csv',parse_dates=["Datetime"],index_col='Datetime')

In [27]:
df_load_new = df_load.tz_localize("Australia/Melbourne", ambiguous=True)

In [28]:
df_load_new.tz_convert('Etc/GMT-10')

,ImportkWh
Datetime,
2021-12-31 23:00:00+10:00,20.836986
2021-12-31 23:30:00+10:00,20.558885
2022-01-01 00:00:00+10:00,20.579485
2022-01-01 00:30:00+10:00,20.600085
2022-01-01 01:00:00+10:00,20.466184
...,...
2022-12-31 20:30:00+10:00,20.620685
2022-12-31 21:00:00+10:00,20.435284
2022-12-31 21:30:00+10:00,20.445584


In [8]:
df_solar

,Datetime,Generation
0,2018-01-01 00:00:00+10:00,0.0
1,2018-01-01 00:30:00+10:00,0.0
2,2018-01-01 01:00:00+10:00,0.0
3,2018-01-01 01:30:00+10:00,0.0
4,2018-01-01 02:00:00+10:00,0.0
...,...,...
17515,2018-12-31 21:30:00+10:00,0.0
17516,2018-12-31 22:00:00+10:00,0.0
17517,2018-12-31 22:30:00+10:00,0.0
17518,2018-12-31 23:00:00+10:00,0.0


In [16]:
df_market.resample('30min').mean()

,ImportWholesalePrice,ExportWholesalePrice
Datetime,,
2024-01-01 00:00:00,96.821667,72.616250
2024-01-01 00:30:00,80.955000,60.716250
2024-01-01 01:00:00,80.370000,60.277500
2024-01-01 01:30:00,57.731667,43.298750
2024-01-01 02:00:00,34.346667,25.760000
...,...,...
2024-12-31 21:30:00,8.940000,11.175000
2024-12-31 22:00:00,19.050000,23.812500
2024-12-31 22:30:00,10.668333,13.335417
